# M5.0c Primer: Components, Transients and AC Circuits

Book pointer: chapter 16, sections 16.1 (circuit components), 16.2 (circuits in the time domain), 16.3 (circuits in the frequency domain), 16.4 (assumptions underlying circuit models).

Third of four primers for module 5. Primers 00a and 00b built direct-current circuits from Ohm's law and Kirchhoff's two laws and ended with a capacitor charging through a resistor. This primer adds the third passive component, the inductor, writes circuits containing all three as differential equations in time, and then shows how a sinusoidal source turns those differential equations into ordinary algebra with complex numbers. It closes with the conditions under which the whole circuit picture is valid at all. Notebook 03 (transients and AC circuits) uses every result derived here, and it calls the same two library routines: `emlib.integrators.rk4_ode` for the time domain and `emlib.circuits.mna` with a frequency for the frequency domain.

## 1. Warm-up

Try these on paper before running anything.

1. Write the voltage-current relation for a resistor, a capacitor and an inductor. Which component stores energy in an electric field, which in a magnetic field, and which stores none?
2. Why can the voltage across a capacitor not change instantaneously? What is the corresponding statement for an inductor?
3. A coil carrying a steady current is suddenly disconnected from its battery. Why can a spark jump across the opening switch?
4. Write the differential equation for the charge on the capacitor of a series RLC circuit with no source. Which mechanical system obeys the same equation?
5. What is the impedance of a capacitor at angular frequency $\omega$? What does it tend to at very low and at very high frequency, and what does that mean physically?
6. At which frequency does the current in a series RLC circuit driven by a sinusoidal source peak, and what sets the height of the peak?
7. Mains voltage in Europe is quoted as 230 V. What is the peak voltage, and why is the RMS value the one that is quoted?

## 2. The ideas

**Three passive components.** A circuit model is built from components that each relate the voltage across them to the current through them. The resistor turns electrical energy into heat at the rate $I^2R$ and stores nothing: its voltage is proportional to its current, $V = RI$, at every instant. The capacitor stores energy in the electric field between its plates. Its charge is proportional to its voltage, $Q = CV$, so its current, the rate at which charge arrives on a plate, is $I = C\,dV/dt$. The inductor stores energy in the magnetic field of a coil. Its voltage is proportional to the rate of change of its current, $V = L\,dI/dt$. Every circuit in this primer is some combination of these three plus sources, and every behaviour that is new compared with primer 00b comes from the two derivatives.

**The inductor, ahead of its physics.** The full explanation needs Faraday's law of induction (chapters 14 and 15, module 6). In short: A current in a coil makes a magnetic field (module 4: the field inside a long solenoid is $\mu_0 nI$). That field threads the turns of the coil itself. When the current changes, the magnetic flux through the turns changes, and a changing flux induces an electromotive force around each turn that opposes the change. The total induced voltage is proportional to $dI/dt$ and the constant of proportionality, the inductance $L$, depends only on the geometry of the coil, exactly as capacitance depends only on the geometry of the plates. The unit is the henry, $1\,\mathrm{H} = 1\,\mathrm{V\,s/A}$. Here the inductor is simply a component that resists changes in its current, as a capacitor resists changes in its voltage.

**Circuits as differential equations.** Kirchhoff's laws still hold: currents into a node sum to zero, and voltages around a loop sum to zero. But the voltage across a capacitor depends on the integral of its current, and the voltage across an inductor on the derivative, so the loop equation becomes a differential equation in time. A circuit with one capacitor or one inductor gives a first-order equation whose solution relaxes exponentially with a time constant $RC$ or $L/R$. A circuit with both gives a second-order equation, the same equation as a mass on a spring with friction: the charge oscillates if the resistance is small and creeps to rest if it is large.

**The sinusoidal steady state.** If the source is a sinusoid, $V_s(t) = V_0\cos\omega t$, then after the transients have died away every voltage and current in a linear circuit is a sinusoid at the same frequency, differing only in amplitude and phase. Writing each sinusoid as the real part of a complex number times $e^{j\omega t}$ turns every time derivative into a multiplication by $j\omega$. The differential equations become complex linear algebra, the same kind of problem as the DC nodal analysis of primer 00b. (Engineers write $j$ for $\sqrt{-1}$ because $i$ is taken by the current; Python writes `1j`.)

**Impedance.** In that complex picture each component obeys a generalised Ohm's law, $\hat V = Z\hat I$, with an impedance $Z$ that depends on frequency: $R$ for a resistor, $1/(j\omega C)$ for a capacitor, $j\omega L$ for an inductor. Impedances combine like resistances. A capacitor is an open circuit at zero frequency and a short at high frequency; an inductor is the reverse.

**Filters and resonance.** Because impedance depends on frequency, a circuit treats different frequencies differently. A resistor and a capacitor in a voltage divider pass low frequencies and block high ones, or the reverse: a filter. Its behaviour is summarised by a transfer function, drawn as a Bode plot. An inductor and a capacitor in series have impedances of opposite sign that cancel exactly at one frequency, $\omega_0 = 1/\sqrt{LC}$. There the circuit is purely resistive and the current peaks: resonance. The sharpness of the peak is measured by the quality factor $Q$, and a high-$Q$ circuit picks one frequency out of many, which is how a radio is tuned.

**When is a circuit a circuit?** A circuit diagram assumes that each component is a point (a lumped element), that the wires are ideal, and that the current is the same at both ends of a wire at the same instant. That last assumption fails when the circuit is not small compared with the distance light travels in one period of the signal, $\lambda = c/f$. At mains frequency $\lambda$ is thousands of kilometres and the assumptions are excellent. At gigahertz frequencies $\lambda$ is centimetres, a trace on a circuit board is a transmission line, and the lumped model must be replaced by the fields themselves (chapter 17 and module 6).

**What notebook 03 does.** It integrates RC, RL and RLC circuits with `rk4_ode` against the closed forms below, solves them in the frequency domain with `mna`, draws Bode plots of single and multi-stage filters, and measures resonance frequency and bandwidth.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import circuits as ci
from emlib import biot_savart as bs
from emlib.integrators import rk4_ode

plt.rcParams["figure.figsize"] = (6, 4)
np.set_printoptions(precision=4, suppress=True)
MU0 = bs.MU0

### 3.1 The three components

**Resistor.** $V = RI$. The voltage and current are proportional at every instant. Power $P = VI = I^2R = V^2/R$ is always positive: a resistor only ever takes energy out of the circuit and turns it into heat (primer 00a, section on energy loss).

**Capacitor.** $Q = CV$, so $I = dQ/dt = C\,dV/dt$. A steady (DC) current cannot flow through a capacitor for long, because charge piles up and the voltage rises without limit; in DC circuits a capacitor in steady state is an open circuit. The energy stored is the work done charging it, $W = \int V\,dQ = \int_0^Q (q/C)\,dq = \tfrac12 CV^2$, and it lives in the electric field between the plates (module 2, primer 00c: energy density $\tfrac12\epsilon_0E^2$).

**Inductor.** $V = L\,dI/dt$, with the sign convention that $V$ is the voltage drop across the inductor in the direction of the current. A steady current flows through an ideal inductor with no voltage drop at all: in DC steady state an inductor is a short circuit. The energy stored is the work done against the induced voltage while the current is built up: the power delivered is $VI = LI\,dI/dt = \frac{d}{dt}\big(\tfrac12LI^2\big)$, so $W = \tfrac12 LI^2$. It lives in the magnetic field of the coil, with density $B^2/2\mu_0$ (module 6).

**Two continuity rules.**

1. *The voltage across a capacitor cannot jump.* A jump in $V$ in zero time would need $dV/dt = \infty$ and so an infinite current $I = C\,dV/dt$. Real circuits have resistance, which limits the current, so $V_C$ is continuous in time.
2. *The current through an inductor cannot jump.* A jump in $I$ would need $dI/dt = \infty$ and so an infinite voltage $V = L\,dI/dt$. So $I_L$ is continuous in time.

Just after a switch is thrown, a capacitor therefore behaves like a voltage source holding its old voltage, and an inductor like a current source holding its old current. Long after, in DC steady state, the capacitor is an open circuit and the inductor a short. Those four statements are enough to sketch any first-order transient before solving anything.

**Where inductance comes from, in one estimate.** For a long solenoid of $N$ turns, length $l$ and cross-section $A$, module 4 gave the field inside as $B = \mu_0 (N/l) I$. The flux through one turn is $BA$ and through all $N$ turns $N BA$. Faraday's law (module 6) says the induced voltage is the rate of change of that total flux, $V = N A\,dB/dt = \mu_0 N^2 A/l\cdot dI/dt$, so
$$L = \frac{\mu_0 N^2 A}{l}.$$
More turns both make more field and catch more of it, hence $N^2$.

**Worked example 1 (a solenoid's inductance and stored energy).** $N = 500$ turns, length $l = 10\,$cm, radius $1\,$cm, so $A = \pi(0.01)^2 = 3.14\times10^{-4}\,\mathrm{m^2}$. Then $L = 4\pi\times10^{-7}\cdot 500^2\cdot3.14\times10^{-4}/0.1 = 9.87\times10^{-4}\,\mathrm{H} \approx 0.99\,$mH. At $I = 2\,$A it stores $\tfrac12LI^2 = 1.97\,$mJ. For comparison, a $100\,\mu$F capacitor at $12\,$V stores $\tfrac12CV^2 = 7.2\,$mJ. The long-solenoid formula assumes the field is $\mu_0 nI$ everywhere inside. The field drops to half at the ends (module 4, notebook 02), so the real coil catches less flux. The code estimates the correction by summing the on-axis field of a finite solenoid over the positions of the turns, which gives about 10 percent less for this coil, whose length is only five times its diameter.

**Worked example 2 (continuity in action).** A $1\,\mu$F capacitor at $0\,$V is connected through a $100\,\Omega$ resistor to a $10\,$V battery. At $t = 0^+$ the capacitor voltage is still $0$, so the whole $10\,$V is across the resistor and the current jumps from $0$ to $0.1\,$A. The capacitor current jumped; its voltage did not. If instead a $10\,$mH inductor carrying $0.5\,$A has its supply switched to a $20\,\Omega$ resistor, at $t = 0^+$ the current is still $0.5\,$A, so the resistor immediately has $10\,$V across it. The inductor voltage jumped; its current did not.

In [ ]:
# Worked example 1: inductance of a solenoid, ideal and with the finite-length correction.
N, l, r = 500, 0.10, 0.01
A = np.pi * r**2
L_ideal = MU0 * N**2 * A / l
z_turns = -l / 2 + (np.arange(N) + 0.5) * l / N                  # positions of the turns along the axis
Bz = bs.solenoid_axis_B(r, l, N / l, 1.0, z_turns)                 # on-axis field at each turn for I = 1 A
L_finite = np.sum(Bz * A)                                          # total flux linkage per ampere
print(f"long-solenoid L = {L_ideal * 1e3:.3f} mH;  summing the finite solenoid's axis field over the turns: {L_finite * 1e3:.3f} mH (ratio {L_finite / L_ideal:.3f})")
print(f"energy at 2 A: {0.5 * L_ideal * 2**2 * 1e3:.2f} mJ;   100 uF capacitor at 12 V: {0.5 * 100e-6 * 12**2 * 1e3:.2f} mJ")

# Worked example 2: the jumps right after switching.
print(f"capacitor case: I(0+) = {10 / 100:.2f} A with V_C(0+) = 0 V;   inductor case: V_R(0+) = {0.5 * 20:.1f} V with I_L(0+) = 0.5 A")

### 3.2 RL circuits: building and breaking a current

A battery $V_0$, a resistor $R$ and an inductor $L$ in series, switched on at $t = 0$ with no current flowing. Kirchhoff's loop law: the battery voltage equals the drops across the resistor and the inductor,
$$V_0 = RI + L\frac{dI}{dt}.$$
This is the same first-order equation as the RC charging equation of primer 00b with the roles of the variables swapped. At $t = 0^+$ the current is still zero (continuity rule 2), so the whole battery voltage appears across the inductor and the current starts rising at the rate $dI/dt = V_0/L$. As the current grows, more of the voltage is taken by the resistor and less is left for the inductor, so the growth slows. In the end $dI/dt = 0$ and $I = V_0/R$: the inductor has become a short circuit. The solution with $I(0) = 0$ is
$$I(t) = \frac{V_0}{R}\left(1 - e^{-t/\tau}\right), \qquad \tau = \frac{L}{R}.$$
Check it: $dI/dt = (V_0/L)e^{-t/\tau}$, and $RI + L\,dI/dt = V_0(1 - e^{-t/\tau}) + V_0e^{-t/\tau} = V_0$. The time constant $L/R$ has units of $\mathrm{(V\,s/A)/(V/A)} = \mathrm{s}$. After one $\tau$ the current has reached $1 - e^{-1} = 63$ percent of its final value, after $5\tau$ more than 99 percent.

**Breaking the current.** Now open the switch. The inductor's current cannot jump, so for a moment the current keeps flowing, and it must flow somewhere. If the circuit is simply broken, the current has nowhere to go and $dI/dt$ becomes enormous, which means $V = L\,dI/dt$ becomes enormous. The voltage across the opening switch contacts rises until the air between them breaks down and the current continues as a spark. This is why switches on motors and relays spark, and how a car's ignition coil makes kilovolts from a 12 V battery. The cure in electronics is a path for the current: a resistor or a diode across the coil (a flyback path). Then the current decays through that path as $I(t) = I_0e^{-t/\tau'}$ with $\tau' = L/R_{\text{total}}$, and the voltage across the coil starts at $I_0R_{\text{path}}$, which is large but finite.

**Worked example 3 (switching on).** $V_0 = 12\,$V, $R = 6\,\Omega$, $L = 0.3\,$H. Final current $V_0/R = 2\,$A, time constant $\tau = 0.3/6 = 0.05\,$s. At $t = \tau$: $I = 2(1 - e^{-1}) = 1.264\,$A. At $t = 5\tau = 0.25\,$s: $I = 1.987\,$A. Final stored energy $\tfrac12\cdot0.3\cdot2^2 = 0.6\,$J. The initial rate of rise is $V_0/L = 40\,$A/s, and if it continued at that rate the current would reach $2\,$A in exactly $\tau$: the time constant is the intercept of the initial tangent with the final value, as for the RC circuit.

**Worked example 4 (switching off).** At $t = 0.25\,$s, with $I = 1.987\,$A, the battery is disconnected. Without a flyback path, if the contacts separate in $1\,\mu$s and the current falls to zero in that time, $V \approx L\,\Delta I/\Delta t = 0.3\cdot2/10^{-6} = 6\times10^5\,$V: far more than enough to spark across a millimetre gap. With a $100\,\Omega$ resistor across the coil the current decays through $106\,\Omega$ in total (the coil's own $6\,\Omega$ plus the $100\,\Omega$), with $\tau' = 0.3/106 = 2.83\,$ms, and the peak voltage across the $100\,\Omega$ resistor is $100\cdot1.987 = 199\,$V, still about 17 times the battery voltage.

In [ ]:
V0, R, L = 12.0, 6.0, 0.3
tau = L / R
t_off, R_fly = 5 * tau, 100.0

# Two stages, integrated separately so that no RK4 step straddles the switch.
t_on = np.linspace(0, t_off, 5001)
I_on = rk4_ode(lambda tk, i: (V0 - R * i) / L, 0.0, t_on)                       # battery connected
t_dec = np.linspace(t_off, t_off + 8 * L / (R + R_fly), 2001)
I_dec = rk4_ode(lambda tk, i: -(R + R_fly) * i / L, I_on[-1], t_dec)            # battery removed: decay through coil and flyback resistor
t = np.concatenate([t_on, t_dec[1:]]); I = np.concatenate([I_on, I_dec[1:]])
on = np.arange(len(t)) < len(t_on)
I_exact_on = V0 / R * (1 - np.exp(-t[on] / tau))
I_off0 = V0 / R * (1 - np.exp(-t_off / tau))
I_exact_off = I_off0 * np.exp(-(t[~on] - t_off) * (R + R_fly) / L)
print(f"tau = {tau:.3f} s;  I(tau) = {np.interp(tau, t, I):.4f} A (exact {V0 / R * (1 - np.exp(-1)):.4f});  I at switch-off = {I_off0:.4f} A")
print(f"max |RK4 - closed form|: on {np.abs(I[on] - I_exact_on).max():.1e} A, off {np.abs(I[~on] - I_exact_off).max():.1e} A")
print(f"peak voltage across the 100 ohm flyback resistor: {R_fly * I_off0:.1f} V from a {V0:.0f} V battery;  decay time L/R_total = {L / (R + R_fly) * 1e3:.2f} ms")
print(f"without a flyback path, current falling 2 A in 1 us: V ~ {L * 2 / 1e-6:.1e} V")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(t, I, label="RK4"); ax[0].plot(t[on], I_exact_on, "k--", lw=1, label="closed form")
ax[0].axvline(tau, color="grey", ls=":"); ax[0].text(tau, 0.2, " t = L/R")
ax[0].set_xlabel("t [s]"); ax[0].set_ylabel("I [A]"); ax[0].legend(); ax[0].set_title("switch on, then off through 100 ohm")
V_L = np.where(on, V0 - R * I, -(R + R_fly) * I)
ax[1].plot(t * 1e3, V_L); ax[1].set_xlim((t_off - 0.01) * 1e3, (t_off + 0.015) * 1e3)
ax[1].set_xlabel("t [ms]"); ax[1].set_ylabel("voltage across the inductor [V]"); ax[1].set_title("the spike at switch-off")
plt.tight_layout(); plt.show()

### 3.3 LC oscillations and the series RLC circuit

**LC: energy sloshing.** Charge a capacitor $C$ to $V_0$ and connect it across an inductor $L$, with no resistance. At first the capacitor holds all the energy, $\tfrac12CV_0^2$, and the current is zero. The capacitor starts to discharge through the inductor, but the inductor will not let the current change quickly, so the current builds up gradually. When the capacitor is fully discharged the current is at its maximum and all the energy is in the inductor's magnetic field. The inductor will not let the current stop either, so it keeps pushing charge, now charging the capacitor the other way round. The energy sloshes back and forth between electric and magnetic form, exactly as the energy of a mass on a spring sloshes between kinetic and potential.

In equations: let $q$ be the charge on the capacitor and $I = dq/dt$ the current flowing onto that plate. The loop law says the capacitor voltage $q/C$ plus the inductor voltage $L\,dI/dt$ sum to zero:
$$L\frac{d^2q}{dt^2} + \frac{q}{C} = 0 \quad\Longrightarrow\quad q(t) = q_0\cos\omega_0t, \qquad \omega_0 = \frac{1}{\sqrt{LC}}.$$
The analogy is exact: $L$ plays the mass, $1/C$ the spring constant, $q$ the displacement and $I$ the velocity. The total energy $\tfrac{q^2}{2C} + \tfrac12LI^2$ is constant, which also gives the peak current: $\tfrac12LI_{\max}^2 = \tfrac12CV_0^2$, so $I_{\max} = V_0\sqrt{C/L}$.

**Series RLC: the damped oscillator.** Add a resistor in the loop. The loop law gains $RI$:
$$L\frac{d^2q}{dt^2} + R\frac{dq}{dt} + \frac{q}{C} = 0.$$
The resistor plays friction. Try $q = e^{st}$: the characteristic equation is $Ls^2 + Rs + 1/C = 0$, with roots
$$s = -\alpha \pm \sqrt{\alpha^2 - \omega_0^2}, \qquad \alpha = \frac{R}{2L}.$$
Three cases, depending on the sign under the square root. With the capacitor charged to $q_0$ and no current at $t = 0$:

- *Underdamped*, $\alpha < \omega_0$ (small $R$). The roots are complex, $s = -\alpha \pm j\omega_d$ with $\omega_d = \sqrt{\omega_0^2 - \alpha^2}$. The charge oscillates inside a decaying envelope: $q(t) = q_0e^{-\alpha t}\big(\cos\omega_dt + \tfrac{\alpha}{\omega_d}\sin\omega_dt\big)$. The envelope decays with time constant $1/\alpha = 2L/R$ and the oscillation is slightly slower than $\omega_0$.
- *Critically damped*, $\alpha = \omega_0$, that is $R = 2\sqrt{L/C}$. A double root $s = -\alpha$, and $q(t) = q_0e^{-\alpha t}(1 + \alpha t)$. The fastest return to zero without overshoot.
- *Overdamped*, $\alpha > \omega_0$ (large $R$). Two real negative roots $s_1, s_2$, and $q(t) = q_0\,\dfrac{s_2e^{s_1t} - s_1e^{s_2t}}{s_2 - s_1}$. No oscillation; the slower root $s_1$ dominates late, so a very large $R$ makes the decay slow again.

Each formula can be checked by substitution: at $t = 0$ each gives $q = q_0$ and $dq/dt = 0$.

**Worked example 5 (an LC circuit).** $L = 10\,$mH, $C = 1\,\mu$F. Then $\omega_0 = 1/\sqrt{10^{-2}\cdot10^{-6}} = 10^4\,$rad/s, $f_0 = \omega_0/2\pi = 1592\,$Hz, period $0.628\,$ms. Charged to $V_0 = 5\,$V it holds $\tfrac12\cdot10^{-6}\cdot25 = 12.5\,\mu$J, and the peak current is $5\sqrt{10^{-6}/10^{-2}} = 0.05\,$A. Check: $\tfrac12\cdot0.01\cdot0.05^2 = 12.5\,\mu$J.

**Worked example 6 (the three regimes).** Same $L$ and $C$, so the critical resistance is $2\sqrt{L/C} = 2\sqrt{10^4} = 200\,\Omega$.
- $R = 20\,\Omega$: $\alpha = 20/0.02 = 1000\,\mathrm{s^{-1}}$, $\omega_d = \sqrt{10^8 - 10^6} = 9950\,$rad/s. The envelope falls by a factor $e$ every $1/\alpha = 1\,$ms, while the period is $2\pi/\omega_d = 0.63\,$ms, so the charge swings about 1.6 times per $e$-folding and has visibly died out after a handful of swings.
- $R = 200\,\Omega$: critical, $\alpha = 10^4\,\mathrm{s^{-1}}$.
- $R = 1000\,\Omega$: $\alpha = 5\times10^4\,\mathrm{s^{-1}}$, $s_{1,2} = -5\times10^4 \pm\sqrt{2.5\times10^9 - 10^8} = -1010$ and $-98\,990\,\mathrm{s^{-1}}$. The slow root makes the overdamped decay take about $1\,$ms, ten times longer than the critical one.

The code integrates all three with `rk4_ode` as a system for $(q, I)$ and compares with the closed forms. It also plots the energy of the LC case moving between the capacitor and the inductor.

In [ ]:
L, C, V0 = 10e-3, 1e-6, 5.0
q0, w0 = C * V0, 1 / np.sqrt(L * C)
print(f"omega0 = {w0:.0f} rad/s, f0 = {w0 / 2 / np.pi:.1f} Hz, period {2 * np.pi / w0 * 1e3:.3f} ms, peak LC current {V0 * np.sqrt(C / L):.3f} A")

def rlc(R):
    return lambda t, y: np.array([y[1], -(R * y[1] + y[0] / C) / L])   # y = (q, I)

def q_exact(R, t):
    a = R / (2 * L)
    if a < w0:
        wd = np.sqrt(w0**2 - a**2)
        return q0 * np.exp(-a * t) * (np.cos(wd * t) + a / wd * np.sin(wd * t))
    if a == w0:
        return q0 * np.exp(-a * t) * (1 + a * t)
    s1, s2 = -a + np.sqrt(a**2 - w0**2), -a - np.sqrt(a**2 - w0**2)
    return q0 * (s2 * np.exp(s1 * t) - s1 * np.exp(s2 * t)) / (s2 - s1)

t = np.linspace(0, 3e-3, 3001)                                    # dt = 1 us, well inside RK4's stability limit for the fast overdamped root
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for R, name in ((20.0, "underdamped, 20 ohm"), (200.0, "critical, 200 ohm"), (1000.0, "overdamped, 1000 ohm")):
    y = rk4_ode(rlc(R), np.array([q0, 0.0]), t)
    err = np.abs(y[:, 0] - q_exact(R, t)).max() / q0
    print(f"{name:22s} alpha = {R / 2 / L:8.0f} 1/s   max |q_RK4 - q_exact| / q0 = {err:.1e}")
    ax[0].plot(t * 1e3, y[:, 0] / q0, label=name)
ax[0].set_xlabel("t [ms]"); ax[0].set_ylabel("q / q0"); ax[0].legend(); ax[0].set_title("series RLC released from a charged capacitor")

y = rk4_ode(rlc(0.0), np.array([q0, 0.0]), t[:1300])
UC, UL = y[:, 0]**2 / (2 * C), 0.5 * L * y[:, 1]**2
ax[1].plot(t[:1300] * 1e3, UC * 1e6, label="capacitor"); ax[1].plot(t[:1300] * 1e3, UL * 1e6, label="inductor")
ax[1].plot(t[:1300] * 1e3, (UC + UL) * 1e6, "k--", label="total")
ax[1].set_xlabel("t [ms]"); ax[1].set_ylabel("energy [uJ]"); ax[1].legend(); ax[1].set_title("LC: energy sloshes between C and L")
plt.tight_layout(); plt.show()
print(f"LC total energy: min {(UC + UL).min() * 1e6:.4f} uJ, max {(UC + UL).max() * 1e6:.4f} uJ (expected 12.5)")

### 3.4 Phasors and impedance

**Why sinusoids are special.** Drive a linear circuit with $V_s(t) = V_0\cos(\omega t)$. Once the transients of sections 3.2 and 3.3 have decayed, every voltage and current oscillates at $\omega$, because differentiating or integrating a sinusoid gives a sinusoid of the same frequency. The only unknowns are amplitudes and phases.

**The complex exponential trick.** Amplitude and phase are exactly what a complex number holds. Write each sinusoid as the real part of a complex exponential:
$$V(t) = |\hat V|\cos(\omega t + \phi) = \mathrm{Re}\big(\hat Ve^{j\omega t}\big), \qquad \hat V = |\hat V|e^{j\phi}.$$
The complex number $\hat V$ is the *phasor*: its length is the amplitude, its angle the phase at $t = 0$. Picture it as an arrow in the complex plane that rotates counter-clockwise at angular speed $\omega$; the physical signal is its shadow on the real axis. The key step is that the time derivative of $\hat Ve^{j\omega t}$ is $j\omega\hat Ve^{j\omega t}$: in phasor language, $d/dt$ becomes multiplication by $j\omega$. Because the circuit equations are linear with real coefficients, one solves with complex exponentials and takes the real part at the end.

**Impedances, derived.** Apply this to each component law.
- Resistor: $V = RI$ gives $\hat V = R\hat I$. Voltage and current in phase.
- Capacitor: $I = C\,dV/dt$ gives $\hat I = j\omega C\hat V$, so $\hat V = \dfrac{1}{j\omega C}\hat I = -\dfrac{j}{\omega C}\hat I$. Since $j = e^{j\pi/2}$, the current *leads* the capacitor voltage by $90^\circ$: the current peaks a quarter period before the voltage does, because the current is what fills the capacitor up.
- Inductor: $V = L\,dI/dt$ gives $\hat V = j\omega L\hat I$. The voltage leads the current by $90^\circ$, or the current *lags*: the inductor only lets the current build after the voltage has been applied.

So each component obeys $\hat V = Z\hat I$ with
$$Z_R = R, \qquad Z_C = \frac{1}{j\omega C}, \qquad Z_L = j\omega L.$$
The impedance is a complex resistance, measured in ohm. Its magnitude is the ratio of amplitudes and its angle is the phase by which voltage leads current. The capacitor's $|Z_C| = 1/\omega C$ is infinite at $\omega = 0$ (open circuit at DC) and goes to zero at high frequency (short circuit). The inductor's $|Z_L| = \omega L$ is zero at DC (short) and grows without limit (open at high frequency).

**Kirchhoff for phasors.** Kirchhoff's laws are linear sums of currents and voltages at a single instant, and the real part of a sum is the sum of real parts, so both laws hold for phasors. Consequently impedances combine like resistances: in series $Z = Z_1 + Z_2$, in parallel $1/Z = 1/Z_1 + 1/Z_2$. Every DC technique of primer 00b carries over with complex numbers; `emlib.circuits.mna` given `omega` builds the nodal equations with admittances $1/Z$ and solves a complex linear system.

Power is a product of two signals, not a linear quantity, so it does not follow from multiplying phasors; section 3.7 gives its rule.

**Worked example 7 (an RC circuit in the frequency domain).** $R = 1\,\mathrm{k\Omega}$ and $C = 1\,\mu$F in series with a source of amplitude $1\,$V at $\omega = 1000\,$rad/s ($f = 159\,$Hz). Then $Z_C = 1/(j\cdot1000\cdot10^{-6}) = -1000j\,\Omega$, and the total impedance is $Z = 1000 - 1000j\,\Omega$, with $|Z| = 1414\,\Omega$ and angle $-45^\circ$. The current phasor is $\hat I = 1/Z = (1000 + 1000j)/(2\times10^6) = (0.5 + 0.5j)\,$mA: amplitude $0.707\,$mA, leading the source by $45^\circ$. The capacitor voltage is $\hat V_C = Z_C\hat I = -1000j(0.5 + 0.5j)\times10^{-3} = 0.5 - 0.5j\,$V: amplitude $0.707\,$V, lagging the source by $45^\circ$. The resistor takes the other $0.5 + 0.5j\,$V, and the two add to the source $1\,$V: Kirchhoff's voltage law, for phasors.

**Worked example 8 (a parallel combination).** $R = 100\,\Omega$ in parallel with $C = 10\,\mu$F at $\omega = 1000\,$rad/s. $Z_C = -100j\,\Omega$. Then $Z = \dfrac{RZ_C}{R + Z_C} = \dfrac{-10^4j}{100 - 100j} = \dfrac{-10^4j(100 + 100j)}{2\times10^4} = 50 - 50j\,\Omega$. Its magnitude, $70.7\,\Omega$, is smaller than either branch, as for resistors, because the two branch currents are $90^\circ$ apart and add like perpendicular vectors.

The code checks both examples three ways: complex arithmetic, `mna` with `omega`, and the long-time solution of the RC differential equation, from which amplitude and phase are fitted.

In [ ]:
R, C, w = 1e3, 1e-6, 1e3
ZC = 1 / (1j * w * C)
I_hat = 1 / (R + ZC); VC_hat = ZC * I_hat
print(f"by hand:  I = {I_hat * 1e3:.4f} mA  (|I| = {abs(I_hat) * 1e3:.4f} mA, phase {np.degrees(np.angle(I_hat)):+.1f} deg);  V_C = {VC_hat:.4f} V")

V, Ivs = ci.mna(3, resistors=[(1, 2, R)], capacitors=[(2, 0, C)], voltage_sources=[(1, 0, 1.0)], omega=w)
print(f"mna:      I = {-Ivs[0] * 1e3:.4f} mA  V_C = {V[2]:.4f} V   (the source current comes out with a minus sign: mna reports it from + to - inside the source)")

# The same steady state from the time domain: dV_C/dt = (V_s(t) - V_C) / (RC), started from rest, run 20 periods.
T = 2 * np.pi / w
t = np.linspace(0, 20 * T, 20 * 400 + 1)
VC = rk4_ode(lambda tk, v: (np.cos(w * tk) - v) / (R * C), 0.0, t)
last = t > 18 * T                                                 # fit the last two periods to a cos + b sin
M = np.column_stack([np.cos(w * t[last]), np.sin(w * t[last])])
a, b = np.linalg.lstsq(M, VC[last], rcond=None)[0]
amp, phase = np.hypot(a, b), np.degrees(np.arctan2(-b, a))        # a cos + b sin = amp cos(wt + phase)
print(f"ODE steady state: amplitude {amp:.5f} V, phase {phase:+.3f} deg;  phasor: {abs(VC_hat):.5f} V, {np.degrees(np.angle(VC_hat)):+.3f} deg")

# Worked example 8.
ZC8 = 1 / (1j * 1000 * 10e-6)
print(f"100 ohm parallel with 10 uF at 1000 rad/s: Z = {100 * ZC8 / (100 + ZC8):.2f} ohm, |Z| = {abs(100 * ZC8 / (100 + ZC8)):.1f} ohm")

plt.plot(t * 1e3, np.cos(w * t), label="source")
plt.plot(t * 1e3, VC, label="capacitor voltage (RK4)")
plt.xlim(0, 6 * T * 1e3); plt.xlabel("t [ms]"); plt.ylabel("V"); plt.legend(); plt.title("the transient dies, the steady state lags by 45 degrees"); plt.show()

### 3.5 Filters and Bode plots

**The RC low-pass filter.** Take the RC circuit of worked example 7 and treat the capacitor voltage as the output. The circuit is a voltage divider with impedances, so the transfer function, the ratio of output phasor to input phasor, is
$$H(\omega) = \frac{\hat V_{\text{out}}}{\hat V_{\text{in}}} = \frac{Z_C}{R + Z_C} = \frac{1/j\omega C}{R + 1/j\omega C} = \frac{1}{1 + j\omega RC}.$$
At low frequency $\omega RC \ll 1$ and $H \approx 1$: the capacitor is nearly an open circuit, no current flows, no voltage is lost across $R$, and the input appears at the output. At high frequency $\omega RC \gg 1$ and $H \approx 1/(j\omega RC)$: the capacitor is nearly a short, and the output falls in proportion to $1/\omega$ with a phase lag approaching $90^\circ$. The crossover is at the *corner frequency* $\omega_c = 1/RC$, where $|H| = 1/\sqrt2$ and the phase is $-45^\circ$. The output power there is half the input power: the half-power or $-3\,$dB point.

**Decibels and the Bode plot.** Gains spanning many orders of magnitude are measured in decibels: $G_{\text{dB}} = 20\log_{10}|H|$ (the factor 20, not 10, because $|H|$ is an amplitude ratio and decibels measure power ratios). $|H| = 1/\sqrt2$ is $-3.01\,$dB, $|H| = 0.1$ is $-20\,$dB, $0.01$ is $-40\,$dB. A Bode plot draws the gain in decibels and the phase against $\log\omega$. On it the low-pass filter is two straight lines: flat at $0\,$dB below $\omega_c$, and falling at $-20\,$dB per decade above it (a tenfold increase of $\omega$ divides $|H|$ by ten). The real curve rounds the corner, $3\,$dB below the asymptotes at $\omega_c$.

**The high-pass filter.** Swap the components: output across the resistor. Then
$$H(\omega) = \frac{R}{R + 1/j\omega C} = \frac{j\omega RC}{1 + j\omega RC},$$
which blocks DC (the capacitor is open) and passes high frequencies, with the same corner $\omega_c = 1/RC$ and a phase that goes from $+90^\circ$ to $0$.

**An RL filter.** An inductor in series with a resistor, output across the resistor: $H = R/(R + j\omega L) = 1/(1 + j\omega L/R)$. A low-pass filter with corner $\omega_c = R/L$. Inductors block high frequencies and pass low ones, the reverse of capacitors.

**Cascading.** Two RC stages in a row do not simply multiply their transfer functions, because the second stage draws current from the first (it *loads* it). Notebook 03 builds a three-stage filter with `mna` and shows the loading directly.

**Worked example 9 (an RC low-pass).** $R = 1\,\mathrm{k\Omega}$, $C = 100\,$nF: $\omega_c = 1/(10^3\cdot10^{-7}) = 10^4\,$rad/s, $f_c = 1592\,$Hz. At $f_c$: $|H| = 0.707$, $-3.01\,$dB, $-45^\circ$. At $10f_c$: $|H| = 1/\sqrt{101} = 0.0995$, $-20.04\,$dB, phase $-\arctan10 = -84.3^\circ$. At $f_c/10$: $|H| = 0.995$, $-0.04\,$dB, $-5.7^\circ$.

**Worked example 10 (an RL low-pass).** $L = 10\,$mH and $R = 100\,\Omega$, output across $R$: $\omega_c = R/L = 10^4\,$rad/s, the same corner as worked example 9. The code shows the two Bode curves on top of each other.

In [ ]:
R, C = 1e3, 100e-9
L, R_L = 10e-3, 100.0
wc = 1 / (R * C)
w = np.logspace(2, 6, 161)

H_rc = np.array([ci.mna(3, resistors=[(1, 2, R)], capacitors=[(2, 0, C)], voltage_sources=[(1, 0, 1.0)], omega=wk)[0][2] for wk in w])
H_rl = np.array([ci.mna(3, inductors=[(1, 2, L)], resistors=[(2, 0, R_L)], voltage_sources=[(1, 0, 1.0)], omega=wk)[0][2] for wk in w])
H_hp = np.array([ci.mna(3, capacitors=[(1, 2, C)], resistors=[(2, 0, R)], voltage_sources=[(1, 0, 1.0)], omega=wk)[0][2] for wk in w])
H_exact = 1 / (1 + 1j * w / wc)
print(f"max |H_mna - 1/(1 + j w RC)| = {np.abs(H_rc - H_exact).max():.1e};   RL vs RC low-pass: {np.abs(H_rl - H_rc).max():.1e}")
for f_ratio in (0.1, 1.0, 10.0):
    H = 1 / (1 + 1j * f_ratio)
    print(f"f = {f_ratio:4.1f} f_c:  |H| = {abs(H):.4f}  gain {20 * np.log10(abs(H)):+7.2f} dB  phase {np.degrees(np.angle(H)):+6.1f} deg")

fig, ax = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
ax[0].semilogx(w, 20 * np.log10(np.abs(H_rc)), lw=3, alpha=0.5, label="RC low-pass (mna)")
ax[0].semilogx(w, 20 * np.log10(np.abs(H_rl)), "k--", lw=1, label="RL low-pass (mna)")
ax[0].semilogx(w, 20 * np.log10(np.abs(H_hp)), label="RC high-pass (mna)")
ax[0].semilogx(w, np.minimum(0, -20 * np.log10(w / wc)), ":", color="grey", label="asymptotes")
ax[0].axvline(wc, color="grey", lw=0.5); ax[0].set_ylabel("gain [dB]"); ax[0].legend(fontsize=8); ax[0].set_ylim(-45, 5)
ax[1].semilogx(w, np.degrees(np.angle(H_rc)), lw=3, alpha=0.5); ax[1].semilogx(w, np.degrees(np.angle(H_hp)))
ax[1].axvline(wc, color="grey", lw=0.5); ax[1].set_ylabel("phase [deg]"); ax[1].set_xlabel("omega [rad/s]")
ax[0].set_title("Bode plot: corner at omega_c = 1/RC = R/L = 1e4 rad/s"); plt.tight_layout(); plt.show()

### 3.6 Resonance in the series RLC circuit

**The impedance.** Drive the series RLC circuit of section 3.3 with a source $V_0\cos\omega t$. The three impedances add:
$$Z(\omega) = R + j\omega L + \frac{1}{j\omega C} = R + j\left(\omega L - \frac{1}{\omega C}\right).$$
The inductor's reactance $\omega L$ is positive and grows with frequency; the capacitor's $-1/\omega C$ is negative and shrinks in magnitude. At one frequency they cancel exactly:
$$\omega L = \frac{1}{\omega C} \quad\Longrightarrow\quad \omega_0 = \frac{1}{\sqrt{LC}},$$
the same frequency at which the LC circuit oscillates freely. There $Z = R$, purely resistive, and the current amplitude $|\hat I| = V_0/|Z|$ reaches its maximum $V_0/R$, in phase with the source. Below $\omega_0$ the capacitor dominates, the circuit is capacitive and the current leads; above, it is inductive and the current lags. Only the resistance limits the response.

**Sharpness: the quality factor.** How narrow the peak is depends on how small $R$ is compared with the reactances at resonance. Define
$$Q = \frac{\omega_0L}{R} = \frac{1}{\omega_0RC} = \frac{1}{R}\sqrt{\frac{L}{C}}.$$
The current falls to $1/\sqrt2$ of its peak (half power) where the reactance equals $\pm R$, that is, where $\omega L - 1/\omega C = \pm R$. The two solutions $\omega_1 < \omega_0 < \omega_2$ are separated by exactly $\omega_2 - \omega_1 = R/L$ (subtract the two quadratic equations), so the full width at half power is
$$\Delta\omega = \frac{R}{L} = \frac{\omega_0}{Q}.$$
A high-$Q$ circuit selects a narrow band of frequencies. $Q$ also measures the free decay: the energy falls as $e^{-Rt/L}$, a fraction $1/Q$ per radian of oscillation.

**Voltage magnification.** At resonance the current is $V_0/R$ and the capacitor voltage is $|\hat I|/(\omega_0C) = V_0/(\omega_0RC) = QV_0$. The inductor voltage is the same magnitude and exactly opposite in phase. In a high-$Q$ circuit the component voltages can be many times the source voltage while cancelling in the sum.

**Worked example 11 (a radio tuning circuit).** An AM receiver uses $L = 250\,\mu$H and a variable capacitor. To receive a station at $f_0 = 1\,$MHz, set $C = 1/(\omega_0^2L) = 1/\big((2\pi\cdot10^6)^2\cdot2.5\times10^{-4}\big) = 101\,$pF. With a coil resistance of $R = 10\,\Omega$: $Q = \omega_0L/R = 6.28\times10^6\cdot2.5\times10^{-4}/10 = 157$. The bandwidth is $f_0/Q = 6.4\,$kHz, about the width of one AM channel (stations are spaced by 9 or 10 kHz), so the circuit picks one station and suppresses its neighbours. A $1\,$mV signal from the antenna appears as $157\,$mV across the capacitor, which is why the tuned circuit also amplifies before any transistor does.

**Worked example 12 (reading off a phase).** In the same circuit at $f = f_0 + 3.18\,$kHz, half a bandwidth above resonance, the reactance is very nearly $+R$, so $Z \approx R(1 + j)$: the current is $1/\sqrt2$ of its peak and lags the source by $45^\circ$. At $f_0 - 3.18\,$kHz it leads by $45^\circ$. ("Very nearly" because the exact half-power frequencies satisfy $\omega_1\omega_2 = \omega_0^2$, so they sit symmetrically about $\omega_0$ on a logarithmic scale, not a linear one. For $Q = 157$ the difference is a part in a thousand, which the code shows.) Across the whole resonance the phase swings from $+90^\circ$ to $-90^\circ$, and the swing is steeper the higher the $Q$.

**Animation.** A series RLC circuit at one driving frequency is shown as three rotating phasors, $\hat V_R$, $\hat V_L$ and $\hat V_C$, drawn head to tail so that they add up to the source phasor, with their shadows on the real axis traced as sine waves. $\hat V_L$ and $\hat V_C$ always point in opposite directions. Then the driving frequency sweeps through resonance: $\hat V_L$ and $\hat V_C$ trade size, cancel exactly at $\omega_0$, where $\hat V_R$ alone equals the source and the current readout peaks, and the phase of the current swings from leading to lagging.

In [ ]:
from emlib import show
show.animation("m5_circuits", "PhasorRLC")

In [ ]:
L, R, f0 = 250e-6, 10.0, 1e6
w0 = 2 * np.pi * f0
C = 1 / (w0**2 * L)
Q = ci.quality_factor(R, L, C)
print(f"C = {C * 1e12:.1f} pF,  resonance_omega = {ci.resonance_omega(L, C):.4e} rad/s (2 pi f0 = {w0:.4e}),  Q = {Q:.1f},  bandwidth f0/Q = {f0 / Q:.0f} Hz")

def series_rlc(wk, Vs=1.0):
    V, Ivs = ci.mna(4, resistors=[(1, 2, R)], inductors=[(2, 3, L)], capacitors=[(3, 0, C)], voltage_sources=[(1, 0, Vs)], omega=wk)
    return -Ivs[0], V[3]                                          # current from the + terminal, voltage across C

f = np.linspace(0.97 * f0, 1.03 * f0, 6001)
I = np.array([series_rlc(2 * np.pi * fk)[0] for fk in f])
k = np.argmax(np.abs(I))
half = np.abs(I) >= np.abs(I).max() / np.sqrt(2)
edges = np.nonzero(np.diff(half.astype(int)))[0]
f_lo = np.interp(1 / np.sqrt(2), np.abs(I[edges[0]:edges[0] + 2]) / np.abs(I).max(), f[edges[0]:edges[0] + 2])
f_hi = np.interp(-1 / np.sqrt(2), -np.abs(I[edges[1]:edges[1] + 2]) / np.abs(I).max(), f[edges[1]:edges[1] + 2])
print(f"peak at {f[k]:.0f} Hz with |I| = {abs(I[k]) * 1e3:.4f} mA (V/R = {1 / R * 1e3:.4f} mA);  half-power band {f_lo:.0f} to {f_hi:.0f} Hz, width {f_hi - f_lo:.0f} Hz")
I0, VC0 = series_rlc(w0, Vs=1e-3)
print(f"at resonance with a 1 mV source: |V_C| = {abs(VC0) * 1e3:.1f} mV = {abs(VC0) / 1e-3:.1f} x source")
for df in (+f0 / Q / 2, -f0 / Q / 2):
    Ik, _ = series_rlc(2 * np.pi * (f0 + df))
    print(f"f0 {df:+.0f} Hz: |I| / peak = {abs(Ik) * R:.4f},  phase of I = {np.degrees(np.angle(Ik)):+.1f} deg")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot((f - f0) / 1e3, np.abs(I) * 1e3); ax[0].axvspan((f_lo - f0) / 1e3, (f_hi - f0) / 1e3, color="grey", alpha=0.2, label="half-power band")
ax[0].set_xlabel("f - f0 [kHz]"); ax[0].set_ylabel("|I| [mA] for 1 V"); ax[0].legend(); ax[0].set_title(f"resonance, Q = {Q:.0f}")
ax[1].plot((f - f0) / 1e3, np.degrees(np.angle(I))); ax[1].set_xlabel("f - f0 [kHz]"); ax[1].set_ylabel("phase of I [deg]"); ax[1].set_title("current leads below, lags above")
plt.tight_layout(); plt.show()

### 3.7 Power in AC circuits

**Instantaneous and average power.** The power delivered to a component is $P(t) = V(t)I(t)$ at every instant. With $V = V_0\cos\omega t$ and $I = I_0\cos(\omega t - \phi)$, where $\phi$ is the angle of the load impedance (the phase by which voltage leads current), a product-to-sum identity gives
$$P(t) = \tfrac12V_0I_0\big[\cos\phi + \cos(2\omega t - \phi)\big].$$
The second term oscillates at twice the driving frequency and averages to zero over a period; it is energy flowing into the capacitors and inductors and back out again each cycle. The first term is the average power, the energy actually delivered and turned into heat or work:
$$\langle P\rangle = \tfrac12V_0I_0\cos\phi.$$
A pure capacitor or inductor has $\phi = \pm90^\circ$ and takes no average power at all: it borrows energy and returns it. A resistor has $\phi = 0$ and takes $\tfrac12V_0I_0 = \tfrac12I_0^2R$.

**RMS values.** The factor $\tfrac12$ is the average of $\cos^2$. It is absorbed by defining the root-mean-square value, $V_{\text{rms}} = \sqrt{\langle V^2\rangle} = V_0/\sqrt2$ for a sinusoid, and likewise for the current. Then $\langle P\rangle = V_{\text{rms}}I_{\text{rms}}\cos\phi$, and for a resistor $\langle P\rangle = I_{\text{rms}}^2R = V_{\text{rms}}^2/R$: the same formulas as DC. So $230\,$V RMS heats a resistor exactly as much as a $230\,$V battery would. The factor $\cos\phi$ is the *power factor*. The product $V_{\text{rms}}I_{\text{rms}}$, the *apparent power* in volt-amperes, is what the wires have to carry; only the fraction $\cos\phi$ of it does work.

**Worked example 13 (mains).** European mains: $230\,$V RMS at $50\,$Hz, so the peak is $230\sqrt2 = 325\,$V. A $2\,$kW kettle is a resistor of $R = V_{\text{rms}}^2/P = 230^2/2000 = 26.5\,\Omega$ and draws $I_{\text{rms}} = 8.70\,$A ($12.3\,$A peak). A motor modelled as $R = 20\,\Omega$ in series with an inductance chosen so that $\omega L = 15\,\Omega$ ($L = 47.7\,$mH) has $|Z| = 25\,\Omega$, power factor $\cos\phi = 20/25 = 0.8$, draws $I_{\text{rms}} = 230/25 = 9.2\,$A, and takes $\langle P\rangle = 9.2^2\cdot20 = 1693\,$W out of an apparent power of $230\cdot9.2 = 2116\,$VA. The code averages $V(t)I(t)$ over a period and gets the same number.

In [ ]:
Vrms, f = 230.0, 50.0
w = 2 * np.pi * f
V0 = Vrms * np.sqrt(2)
print(f"peak mains voltage {V0:.1f} V;  2 kW kettle: R = {Vrms**2 / 2000:.2f} ohm, I_rms = {2000 / Vrms:.3f} A, I_peak = {2000 / Vrms * np.sqrt(2):.2f} A")

R, L = 20.0, 15.0 / w
Z = R + 1j * w * L
I0 = V0 / Z                                                         # current phasor for a source phasor V0 (real)
t = np.linspace(0, 1 / f, 20001)
v = V0 * np.cos(w * t)
i = np.real(I0 * np.exp(1j * w * t))
P_avg = np.trapezoid(v * i, t) * f
print(f"motor: L = {L * 1e3:.1f} mH, |Z| = {abs(Z):.1f} ohm, power factor {np.cos(np.angle(Z)):.3f}, I_rms = {abs(I0) / np.sqrt(2):.2f} A")
print(f"average of v(t) i(t) over a period: {P_avg:.1f} W;  V_rms I_rms cos(phi) = {Vrms * abs(I0) / np.sqrt(2) * np.cos(np.angle(Z)):.1f} W;  apparent power {Vrms * abs(I0) / np.sqrt(2):.0f} VA")

plt.plot(t * 1e3, v * i / 1e3, label="p(t) = v(t) i(t)")
plt.axhline(P_avg / 1e3, color="k", ls="--", label="average")
plt.xlabel("t [ms]"); plt.ylabel("power [kW]"); plt.legend(); plt.title("power factor 0.8: part of each cycle flows back to the source"); plt.show()

### 3.8 Assumptions underlying circuit models

A circuit diagram is a model of the electromagnetic field, obtained in chapters 9 and 10 by assuming that the fields are static or change slowly. Every result of this primer rests on four assumptions.

1. **Lumped elements.** Each component is treated as a point with two terminals. All electric field energy is in the capacitors, all magnetic energy in the inductors, all dissipation in the resistors; the wires are ideal.
2. **Kirchhoff's current law holds instant by instant.** The current entering a wire at one end leaves at the other at the same moment. That holds only if a signal crosses the wire in a time negligible compared with the time scale on which the current changes.
3. **Kirchhoff's voltage law holds.** The sum of voltages around a loop is zero only if $\oint\mathbf{E}\cdot d\mathbf{l} = 0$, which is false when the magnetic flux through the loop changes (Faraday's law, module 6). The model puts all changing flux inside the inductors; a large wire loop with a fast-changing current is an inductor the diagram does not show.
4. **No radiation.** A circuit whose currents change fast enough radiates electromagnetic waves (chapter 17) and loses energy that no component in the diagram accounts for.

All four conditions come down to one number: the size of the circuit compared with the wavelength $\lambda = c/f$ of a signal at the frequencies involved. A circuit much smaller than $\lambda$, say less than $\lambda/10$, is well described by lumped elements. Near $\lambda$, current and voltage vary along each wire, which becomes a transmission line.

**Worked example 14 (when wires stop being wires).** At $50\,$Hz, $\lambda = 3\times10^8/50 = 6000\,$km: every household circuit is tiny in comparison, and the lumped model is excellent. At $1\,$MHz (AM radio), $\lambda = 300\,$m: a circuit board is fine, the antenna is not. At $1\,$GHz, $\lambda = 30\,$cm, and a $10\,$cm trace on a circuit board is a third of a wavelength: it is no longer a wire, and high-speed boards are designed with controlled-impedance transmission lines.

**Real components.** Even within the lumped regime, real components are not ideal. A coil has resistance (the $R$ in every RL example above is mostly the coil's own wire) and capacitance between its turns, so it resonates by itself; a capacitor has lead inductance, so above its own resonance it behaves like an inductor. These *parasitic* elements are modelled by adding small components to the diagram, valid while the circuit stays small compared with $\lambda$.

In [ ]:
c = 299792458.0
sizes = {"house wiring (20 m)": 20.0, "circuit board (10 cm)": 0.10, "chip (1 cm)": 0.01}
print(f"{'frequency':>12s} {'wavelength':>12s}   size / wavelength for: " + ", ".join(sizes))
for fk, name in ((50, "50 Hz"), (1e6, "1 MHz"), (1e8, "100 MHz"), (1e9, "1 GHz"), (5e9, "5 GHz")):
    lam = c / fk
    ratios = "  ".join(f"{s / lam:9.1e}{' *' if s / lam > 0.1 else '  '}" for s in sizes.values())
    print(f"{name:>12s} {lam:10.3g} m   {ratios}")
print("* : larger than a tenth of a wavelength, where the lumped circuit model starts to fail")

### Exercise: the low-pass transfer function

Replace the body of `rc_lowpass_H` so that it returns the complex transfer function $\hat V_{\text{out}}/\hat V_{\text{in}}$ of an RC low-pass filter (output across the capacitor) at angular frequency `omega`. Derive it from the voltage divider with impedances before writing any code. The check compares your function with the nodal solver at three frequencies spanning the corner, in both magnitude and phase.

In [ ]:
def rc_lowpass_H(omega, R, C):
    '''Complex transfer function of an RC low-pass filter, output across the capacitor.'''
    ZC = 1 / (1j * omega * C)                                      # replace from here with your own derivation
    return ZC / (R + ZC)


R, C = 4.7e3, 22e-9
for wk in (0.1 / (R * C), 1 / (R * C), 30 / (R * C)):
    H_mna = ci.mna(3, resistors=[(1, 2, R)], capacitors=[(2, 0, C)], voltage_sources=[(1, 0, 1.0)], omega=wk)[0][2]
    H_you = rc_lowpass_H(wk, R, C)
    print(f"omega RC = {wk * R * C:5.1f}:  yours {H_you:.5f}   mna {H_mna:.5f}")
    assert abs(H_you - H_mna) < 1e-12
print("exercise passed")

## 4. Verify

Two closed-form checks with asserts.

1. *Resonance.* For a series RLC circuit with $L = 10\,$mH, $C = 1\,\mu$F and $R = 20\,\Omega$ ($\omega_0 = 10^4\,$rad/s, $Q = 5$), sweep the frequency with `mna`, find the peak of the current and the two half-power frequencies, and compare with $\omega_0 = 1/\sqrt{LC}$, the peak $V_0/R$ and the width $\omega_0/Q = R/L$.
2. *Underdamped transient.* Integrate the same circuit released from a charged capacitor with `rk4_ode` and compare with $q_0e^{-\alpha t}(\cos\omega_dt + \tfrac{\alpha}{\omega_d}\sin\omega_dt)$ over ten periods.

In [ ]:
L, C, R = 10e-3, 1e-6, 20.0
w0, Q = ci.resonance_omega(L, C), ci.quality_factor(R, L, C)
w = np.linspace(0.5 * w0, 1.5 * w0, 20001)
I = np.abs([ci.mna(4, resistors=[(1, 2, R)], inductors=[(2, 3, L)], capacitors=[(3, 0, C)], voltage_sources=[(1, 0, 1.0)], omega=wk)[1][0] for wk in w])
k = int(np.argmax(I))
above = I >= I.max() / np.sqrt(2)
i1, i2 = np.nonzero(above)[0][[0, -1]]
w1 = np.interp(I.max() / np.sqrt(2), I[i1 - 1:i1 + 1], w[i1 - 1:i1 + 1])
w2 = np.interp(-I.max() / np.sqrt(2), -I[i2:i2 + 2], w[i2:i2 + 2])
print(f"peak at omega/omega0 = {w[k] / w0:.5f};  peak |I| R = {I[k] * R:.6f};  bandwidth (w2 - w1) / (omega0 / Q) = {(w2 - w1) / (w0 / Q):.5f}")
assert abs(w[k] / w0 - 1) < 1e-4 and abs(I[k] * R - 1) < 1e-9 and abs((w2 - w1) / (w0 / Q) - 1) < 1e-3

q0, a = C * 1.0, R / (2 * L)
wd = np.sqrt(w0**2 - a**2)
t = np.linspace(0, 10 * 2 * np.pi / wd, 6001)
y = rk4_ode(lambda tk, s: np.array([s[1], -(R * s[1] + s[0] / C) / L]), np.array([q0, 0.0]), t)
q_ex = q0 * np.exp(-a * t) * (np.cos(wd * t) + a / wd * np.sin(wd * t))
err = np.abs(y[:, 0] - q_ex).max() / q0
print(f"underdamped transient over ten periods: max |q_RK4 - q_exact| / q0 = {err:.1e}")
assert err < 1e-6
print("verify passed")

## 5. Explore and rebuild

**Explore.** Build a parallel RLC circuit (the three components side by side, driven by a current source) with `mna`. Show that its *voltage* peaks at the same $\omega_0 = 1/\sqrt{LC}$, but that now a *large* resistance gives a sharp peak, so that $Q = R\sqrt{C/L}$, the inverse of the series formula. Explain the difference in terms of which component the current prefers at frequencies away from resonance. Then add a $1\,\Omega$ series resistance to the inductor branch, as a real coil has, and find how far the peak moves.

**Rebuild.** Close this notebook. On paper, derive the three impedances from the component laws, the low-pass transfer function and its corner frequency, the series RLC resonance condition and the bandwidth $\Delta\omega = R/L$, and the three damping regimes from the characteristic equation. In a blank editor, write a time-domain RLC integrator with `rk4_ode` and a frequency sweep with `mna`, and reproduce the numbers of worked examples 6 and 11.

## 6. Key results

| result | formula | holds when |
|---|---|---|
| resistor | $V = RI$, $P = I^2R$ | ohmic material |
| capacitor | $I = C\,dV/dt$, $W = \tfrac12CV^2$ | linear capacitor |
| inductor | $V = L\,dI/dt$, $W = \tfrac12LI^2$ | linear inductor, no saturation |
| long solenoid | $L = \mu_0N^2A/l$ | length much greater than radius |
| continuity | $V_C$ and $I_L$ continuous in time | any finite voltage and current |
| DC steady state | capacitor open, inductor short | after transients |
| RC charging | $V_C = V_0(1 - e^{-t/RC})$ | step from rest |
| RL rise | $I = (V_0/R)(1 - e^{-tR/L})$, $\tau = L/R$ | step from rest |
| LC oscillation | $\omega_0 = 1/\sqrt{LC}$, $I_{\max} = V_0\sqrt{C/L}$ | no resistance |
| series RLC | $Lq'' + Rq' + q/C = 0$, $\alpha = R/2L$ | |
| regimes | under $\alpha < \omega_0$, critical $R = 2\sqrt{L/C}$, over $\alpha > \omega_0$ | |
| underdamped | $q = q_0e^{-\alpha t}(\cos\omega_dt + \frac{\alpha}{\omega_d}\sin\omega_dt)$, $\omega_d = \sqrt{\omega_0^2 - \alpha^2}$ | $q(0) = q_0$, $I(0) = 0$ |
| phasor | $V(t) = \mathrm{Re}(\hat Ve^{j\omega t})$, $d/dt \to j\omega$ | sinusoidal steady state, linear circuit |
| impedances | $Z_R = R$, $Z_C = 1/j\omega C$, $Z_L = j\omega L$ | |
| combining | series $Z_1 + Z_2$, parallel $1/Z = 1/Z_1 + 1/Z_2$ | |
| RC low-pass | $H = 1/(1 + j\omega RC)$, $\omega_c = 1/RC$, $-3\,$dB, $-45^\circ$, $-20\,$dB/decade | output across $C$, unloaded |
| RC high-pass | $H = j\omega RC/(1 + j\omega RC)$ | output across $R$, unloaded |
| RL low-pass | $H = 1/(1 + j\omega L/R)$, $\omega_c = R/L$ | output across $R$ |
| decibels | $G = 20\log_{10}|H|$ | amplitude ratio |
| series resonance | $Z = R$ at $\omega_0$, $|I|_{\max} = V_0/R$ | |
| quality factor | $Q = \omega_0L/R = \frac1R\sqrt{L/C}$, $\Delta\omega = \omega_0/Q = R/L$, $|V_C| = QV_0$ at $\omega_0$ | series RLC |
| average power | $\langle P\rangle = V_{\text{rms}}I_{\text{rms}}\cos\phi$, $V_{\text{rms}} = V_0/\sqrt2$ | sinusoidal |
| lumped model | circuit size $\ll \lambda = c/f$ (rule of thumb $< \lambda/10$) | |

## 7. Warm-up answers

1. Resistor $V = RI$, capacitor $I = C\,dV/dt$, inductor $V = L\,dI/dt$. The capacitor stores energy in the electric field, the inductor in the magnetic field, and the resistor stores none: it converts energy to heat.
2. A jump in $V_C$ would need an infinite current $C\,dV/dt$, which no real circuit can supply. For the same reason a jump in $I_L$ would need an infinite voltage $L\,dI/dt$, so the inductor current cannot change instantaneously.
3. The inductor's current cannot stop instantly. When the circuit is broken, $dI/dt$ becomes very large, and so does $V = L\,dI/dt$. The voltage across the opening contacts rises until the air breaks down and the current continues as a spark.
4. $L\,d^2q/dt^2 + R\,dq/dt + q/C = 0$. A mass on a spring with friction: $L$ is the mass, $R$ the friction coefficient and $1/C$ the spring constant.
5. $Z_C = 1/(j\omega C)$, magnitude $1/\omega C$. It is infinite at $\omega \to 0$ (an open circuit at DC, since a steady current cannot flow through the gap) and goes to zero at high frequency (a short circuit, since the plates charge and discharge before any voltage builds up).
6. At $\omega_0 = 1/\sqrt{LC}$, where the inductor and capacitor reactances cancel. The impedance is then just $R$, so the peak current is $V_0/R$: the resistance sets the height, and its ratio to $\sqrt{L/C}$ sets the width.
7. The peak is $230\sqrt2 = 325\,$V. RMS is quoted because the average power delivered to a resistor is $V_{\text{rms}}^2/R$, the same as a DC supply of that voltage, so RMS values make AC and DC power calculations identical.